In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import data_analysis
import numpy as np

## Settings

In [ ]:
DATA_FILE = "input_data.csv"
PARAMETER_COLUMNS_SETS = [
    [
        "G_VH_dB",
        "G_VV_dB",
        "projectedLocalIncidenceAngle",
        "assumed_silt_and_clay_content",
    ],
    [
        "C11_ILSF9_BI1000_dB",
        "C22_ILSF9_BI1000_dB",
        "Surface_r_ILSF9_BI1000",
        "Volume_g_ILSF9_BI1000",
        "Ratio_b_ILSF9_BI1000",
        "projectedLocalIncidenceAngle",
        "assumed_silt_and_clay_content",
    ],
]
PARAMETERS_SET = 0 # set of parameters from the list above, counting from 0 for the first one
COMMENT = "soil" # string or False (empty string works as False)
SELECTED_PARAMETERS = PARAMETER_COLUMNS_SETS[PARAMETERS_SET]
TARGET = "soil_moisture"
SPLIT_RANDOM_STATE = 727 # integer or None for random split
TUNE_HYPERPARAMETERS = True # True or False to skip
# Model parameters to use without tuning
KERNEL = "rbf"
C = 2950
EPSILON = 2
GAMMA = 1.35
# Dictionary with parameters to tune
# (will be used only if TUNE_HYPERPARAMETERS = True)
HYPERPARAMETERS_GRID = {
    "svr__kernel": ["rbf"],
    "svr__C": [1, 10, 25, 50, 100, 500, 1000],
    "svr__epsilon": [1, 2, 3, 4, 5],
    "svr__gamma": [0, 0.1, 0.25, 0.5, 0.75, 1, 1.25, 1.5],
}
LIMIT_DATASET = None # Number of rows or None to skip
EXPORT_MODEL = False # True or False to skip
PLOT_MODEL_PARAMETERS_INFLUENCE = False # True or False to skip

In [ ]:
if not SPLIT_RANDOM_STATE:
    from random import randint

    SPLIT_RANDOM_STATE = randint(1, 1000)
    print(f"Random split state: {SPLIT_RANDOM_STATE}")

## Helper functions

In [ ]:
from typing import Tuple

def rounded_range(data: pd.Series, resolution: int = 10) -> Tuple[int, int]:
    bottom = round(data.min() / resolution - 0.5) * resolution
    top = round(data.max() / resolution + 0.5) * resolution

    return (bottom, top)

## Set default font for graphs

In [ ]:
mpl.rcParams["font.family"] = "Palatino Linotype"

## Read data

In [ ]:
df = pd.read_csv(DATA_FILE, sep=",", engine="python")
df = df[[TARGET] + SELECTED_PARAMETERS]
df.head()

In [ ]:
if LIMIT_DATASET:
    df = df.iloc[0:LIMIT_DATASET]
    df

## Exploratory data analysis

In [ ]:
if COMMENT:
    corr_file_name = f"svr_correlations_set_{PARAMETERS_SET}_{COMMENT}.png"
else:
    corr_file_name = f"svr_correlations_set_{PARAMETERS_SET}.png"

data_analysis.correlation_matrix_heatmap(df, output_file=corr_file_name)

## Prepare data for training

In [ ]:
df.dropna(inplace=True)
df.reset_index(inplace=True, drop=True)

In [ ]:
bins_resolution = 15
_, top_range = rounded_range(df[TARGET], resolution = bins_resolution)
bins = list(range(0, top_range + bins_resolution, bins_resolution))
labels = [f"{bins[i]}−{bins[i+1]}" for i in range(len(bins) - 1)]
df["moisture_bin"] = pd.cut(df[TARGET], bins=bins, labels=labels)

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SPLIT_RANDOM_STATE)

for train_index, validation_index in split.split(df, df["moisture_bin"]):
    df_stratified_training = df.loc[train_index]
    df_stratified_validation = df.loc[validation_index]

In [ ]:
bin_counts = df_stratified_training["moisture_bin"].value_counts()
bin_counts = bin_counts.sort_index()

plt.figure(figsize=(8, 6), dpi=300)
bars = plt.bar(bin_counts.index.astype(str), bin_counts.values)

for i, bar in enumerate(bars):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        10,
        str(bin_counts.values[i]),
        ha="center",
        va="bottom",
        fontsize=16,
        rotation=90,
        color="black",
    )

plt.title("Measurements counts in soil moisture bins", fontsize=16)
plt.xlabel("Soil moisture bin (%)", fontsize=16)
plt.ylabel("Measurements count", fontsize=16)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis="y", linestyle='--', alpha=0.7)
plt.tight_layout()

if COMMENT:
    distribution_file_name = f"svr_distribution_set_{PARAMETERS_SET}_{COMMENT}.png"
else:
    distribution_file_name = f"svr_distribution_set_{PARAMETERS_SET}.png"
plt.savefig(distribution_file_name)

plt.show()

In [ ]:
for dataset in [df_stratified_training, df_stratified_validation]:
    dataset.drop(columns=["moisture_bin", "sample_weight"], inplace=True, errors="ignore")

In [ ]:
X_training, y_training = data_analysis.split_data(df_stratified_training, TARGET)
X_validation, y_validation = data_analysis.split_data(df_stratified_validation, TARGET)

## Train model

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

In [ ]:
if TUNE_HYPERPARAMETERS:
    from sklearn.model_selection import GridSearchCV

    svr_pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=False, with_std=True)),
        ("svr", SVR())
    ])

    grid_search = GridSearchCV(
        estimator=svr_pipe,
        param_grid=HYPERPARAMETERS_GRID,
        cv=5,
        scoring="r2",  # "r2" or "neg_mean_squared_error"
        n_jobs=-1,
        verbose=2
    )
    
    grid_search.fit(X_training, y_training)

    print("Best parameters:", grid_search.best_params_)

    svr_model = grid_search.best_estimator_
else:
    print("Skipped parameters tuning")
    
    svr_model = Pipeline([
        ("scaler", StandardScaler(with_mean=False, with_std=True)),
        ("svr", SVR(kernel=KERNEL, C=C, epsilon=EPSILON, gamma=GAMMA))
    ])
    
    svr_model.fit(X_training, y_training)

In [ ]:
if TUNE_HYPERPARAMETERS:
    C = grid_search.best_params_["svr__C"]
    EPSILON = grid_search.best_params_["svr__epsilon"]
    GAMMA = grid_search.best_params_["svr__gamma"]

In [ ]:
if EXPORT_MODEL:
    import joblib
    
    if COMMENT:
        model_file_name = f"svr_model_set_{PARAMETERS_SET}_{COMMENT}.pkl"
    else:
        model_file_name = f"svr_model_set_{PARAMETERS_SET}.pkl"
    
    joblib.dump(rf_model, model_file_name)

## Analyze model performance

In [ ]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd

In [ ]:
svr_pred = svr_model.predict(X_validation)

rmse = root_mean_squared_error(y_validation, svr_pred)
r_squared = svr_model.score(X_validation, y_validation)

print("Performance for unknown data:")
print(f"Root mean Squared Error: {rmse:.2f}")
print(f"R-squared value: {r_squared:.2f}")

In [ ]:
svr_pred_training = svr_model.predict(X_training)

rmse_training = root_mean_squared_error(y_training, svr_pred_training)
r_squared_training = svr_model.score(X_training, y_training)

print("Performance for known data:")
print(f"Mean Squared Error: {rmse_training:.2f}")
print(f"R-squared value: {r_squared_training:.2f}")

In [ ]:
_, top_y = rounded_range(y_validation, resolution=10)
_, top_svr_pred = rounded_range(svr_pred, resolution=10)

axis_min = 0
axis_max = max(top_y, top_svr_pred)

plt.figure(figsize=(10, 8), dpi=300)

sns.scatterplot(x=y_training, y=svr_model.predict(X_training), color="red", label="Predictions on train dataset")
sns.scatterplot(x=y_validation, y=svr_pred, color="blue", label="Predictions on validation dataset")
p = sns.regplot(x=y_validation, y=svr_pred, scatter=False, color="blue", label="Regression line (validation)")
slope, intercept, r, p, sterr = scipy.stats.linregress(x=p.get_lines()[0].get_xdata(), y=p.get_lines()[0].get_ydata())

plt.plot([axis_min, axis_max], [axis_min, axis_max], "r--", label="Perfect prediction")

plt.xlabel("Measured soil moisture (%)", fontsize=16)
plt.ylabel("Predicted soil moisture (%)", fontsize=16)
plt.title(f"Support vector regression prediction of soil moisture\nRMSE: {rmse:.2f}, R²: {r_squared:.2f}, y = {slope:.3f} x + {intercept:.3f}", fontsize=16, fontweight="bold")
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.legend(fontsize=16)
plt.grid(True)
plt.xlim(axis_min, axis_max)
plt.ylim(axis_min, axis_max)
plt.gca().set_aspect("equal", adjustable="box")
plt.tight_layout()

if COMMENT:
    prediction_test_file_name = f"svr_prediction_set_{PARAMETERS_SET}_{COMMENT}.png"
else:
    prediction_test_file_name = f"svr_prediction_set_{PARAMETERS_SET}.png"

plt.savefig(prediction_test_file_name)
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(svr_model, X_training, y_training, cv=5, scoring="neg_root_mean_squared_error")
print(f"Cross-validated RMSE: {-scores.mean():.2f} +/- {scores.std():.2f}")

In [ ]:
if PLOT_MODEL_PARAMETERS_INFLUENCE:
    train_errors = []
    test_errors = []
    
    var_range = range(10, 10020, 250)
    
    for c in var_range:
        model = Pipeline([
            ("scaler", StandardScaler(with_mean=False, with_std=True)),
            ("svr", SVR(kernel=KERNEL, C=c, epsilon=EPSILON, gamma=GAMMA))
        ])
        
        model.fit(X_training, y_training)
        
        train_pred = model.predict(X_training)
        test_pred = model.predict(X_validation)
        
        train_rmse = root_mean_squared_error(y_training, train_pred)
        test_rmse = root_mean_squared_error(y_validation, test_pred)
        
        train_errors.append(train_rmse)
        test_errors.append(test_rmse)
    plt.plot(var_range, train_errors, label="Train data")
    plt.plot(var_range, test_errors, label="Test data")
    plt.xlabel("C")
    plt.ylabel("RMSE")
    plt.title("RMSE C")
    plt.legend()
    plt.show()

In [ ]:
if PLOT_MODEL_PARAMETERS_INFLUENCE:
    train_errors = []
    test_errors = []
    
    var_range = [x / 100.0 for x in range(5, 1005, 50)]
    
    for epsilon in var_range:
        model = Pipeline([
            ("scaler", StandardScaler(with_mean=False, with_std=True)),
            ("svr", SVR(kernel=KERNEL, C=C, epsilon=epsilon, gamma=GAMMA))
        ])
        
        model.fit(X_training, y_training)
        
        train_pred = model.predict(X_training)
        test_pred = model.predict(X_validation)
        
        train_rmse = root_mean_squared_error(y_training, train_pred)
        test_rmse = root_mean_squared_error(y_validation, test_pred)
        
        train_errors.append(train_rmse)
        test_errors.append(test_rmse)
    plt.plot(var_range, train_errors, label="Train data")
    plt.plot(var_range, test_errors, label="Test data")
    plt.xlabel("epsilon")
    plt.ylabel("RMSE")
    plt.title("RMSE epsilon")
    plt.legend()
    plt.show()

In [ ]:
if PLOT_MODEL_PARAMETERS_INFLUENCE:
    train_errors = []
    test_errors = []
    
    var_range = [x / 100.0 for x in range(1, 1005, 50)]
    
    for gamma in var_range:
        model = Pipeline([
            ("scaler", StandardScaler(with_mean=False, with_std=True)),
            ("svr", SVR(kernel=KERNEL, C=C, epsilon=EPSILON, gamma=gamma))
        ])
        
        model.fit(X_training, y_training)
        
        train_pred = model.predict(X_training)
        test_pred = model.predict(X_validation)
        
        train_rmse = root_mean_squared_error(y_training, train_pred)
        test_rmse = root_mean_squared_error(y_validation, test_pred)
        
        train_errors.append(train_rmse)
        test_errors.append(test_rmse)
    plt.plot(var_range, train_errors, label="Train data")
    plt.plot(var_range, test_errors, label="Test data")
    plt.xlabel("gamma")
    plt.ylabel("RMSE")
    plt.title("RMSE gamma")
    plt.legend()
    plt.show()

## Residuals analysis

In [ ]:
residuals = y_validation - svr_pred

In [ ]:
print("Residuals statistics:")
print(f"Mean: {float(np.mean(residuals)):.2f}")
print(f"MAE: {float(np.mean(np.abs(residuals))):.2f}")
print(f"RMSE: {float(np.sqrt(np.mean(residuals**2))):.2f}")
print(f"Skew: {float(scipy.stats.skew(residuals, bias=False)):.2f}")

In [ ]:
plt.figure(figsize=(10, 8), dpi=300)

bin_edges = np.arange(residuals.min() - (residuals.min() % 2), residuals.max() + 2, 2)

plt.hist(residuals, bins=bin_edges, density=True, color="blue", rwidth=0.99)

plt.title("Support vector regression - histogram of residuals", fontsize=16, fontweight="bold")
plt.xlabel("Residual", fontsize=16)
plt.ylabel("Density", fontsize=16)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.xlim(-20, 20)
plt.grid(visible=True, axis="y")
plt.tight_layout()

if COMMENT:
    residuals_file_name = f"svr_residuals_hist_{PARAMETERS_SET}_{COMMENT}.png"
else:
    residuals_file_name = f"svr_residuals_hist_{PARAMETERS_SET}.png"

plt.savefig(residuals_file_name)
plt.show()

## Generate output file

In [ ]:
if COMMENT:
    text_file_name = f"svr_parameters_set_{PARAMETERS_SET}_{COMMENT}.txt"
else:
    text_file_name = f"svr_parameters_set_{PARAMETERS_SET}.txt"

with open(text_file_name, "w") as f:
    f.write("Input parameters:\n")
    for parameter in SELECTED_PARAMETERS:
        f.write(f"{parameter}\n")
    if COMMENT:
        f.write(f"\nComment: {COMMENT}\n")
    f.write("\nModel parameters:\n")
    f.write(f"C: {C}\n")
    f.write(f"epsilon: {EPSILON}\n")
    f.write(f"gamma: {GAMMA}\n\n")
    f.write("Performance for unknown data:\n")
    f.write(f"Root Mean Squared Error: {rmse:.2f}\n")
    f.write(f"R-squared value: {r_squared:.2f}\n")
    f.write("\nPerformance for known data:\n")
    f.write(f"Root Mean Squared Error: {rmse_training:.2f}\n")
    f.write(f"R-squared value: {r_squared_training:.2f}\n")
    f.write(f"\nCross-validated RMSE: {-scores.mean():.2f} +/- {scores.std():.2f}\n")
    f.write("\nResiduals statistics:\n")
    f.write(f"Mean: {float(np.mean(residuals)):.2f}\n")
    f.write(f"MAE: {float(np.mean(np.abs(residuals))):.2f}\n")
    f.write(f"RMSE: {float(np.sqrt(np.mean(residuals**2))):.2f}\n")
    f.write(f"Skew: {float(scipy.stats.skew(residuals, bias=False)):.2f}\n")